# Passo 7 – Comparação de candles pelo lucro (Fase B, **só comparação**)

O candle foi escolhido na Fase A pelo R². Aqui, **sem mudar essa escolha**, perguntamos: se tivéssemos escolhido o candle olhando o dinheiro, o que veríamos?

Para cada candle usamos a sua melhor combinação de janela e re-treino pelo R² (Passo 4, RF padrão) e medimos em 2024, com custo de 5 pts:
* operar sempre e operar só quando o previsto > custo;
* o **acerto de direção necessário só para empatar**, se ganhos e perdas tivessem o tamanho médio do movimento M: p·M − (1 − p)·M − 5 = 0 → **p = (1 + 5 ÷ M) ÷ 2**.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rf_zero as rz
g4 = pd.read_csv("saidas/A4_grade.csv", index_col=[0, 1, 2])
prevs = pd.read_pickle("saidas/A4_previsoes.pkl")
linhas = {}
for m in [1, 5, 15, 30, 60]:
    k = g4.xs(m, level="candle")["R² %"].idxmax(); chave = (m, *k); d = prevs[chave]
    pts = rz.prev_pontos(d); M = d["mov_prox"].abs().mean()
    linha = {"janela": k[0], "re-treino": k[1], "R² %": g4.loc[chave, "R² %"], "acerto direção %": g4.loc[chave, "acerto direção %"],
             "mov. médio M (pts)": M, "acerto p/ empatar %": 100 * (1 + rz.CUSTO_PTS / M) / 2}
    for rot, dirc in [("sempre", np.sign(pts)), ("previsto > custo", np.sign(pts).where(pts.abs() > rz.CUSTO_PTS, 0))]:
        op = (dirc != 0).values
        res = rz.resultado_operacoes(dirc.values, d["mov_prox"].values)
        linha[f"{rot}: ops"] = op.sum()
        linha[f"{rot}: pts/op"] = res.mean() if len(res) else np.nan
        linha[f"{rot}: total R$"] = res.sum() * rz.REAIS_POR_PONTO["WIN"]
    linhas[f"{m} min"] = linha
comp = pd.DataFrame(linhas).T
comp.to_csv("saidas/B7_comparacao_candles.csv")
comp

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
axs[0].bar(comp.index, comp["acerto direção %"].astype(float), color="#2a7ab9", label="acerto do modelo")
axs[0].plot(comp.index, comp["acerto p/ empatar %"].astype(float), "o-", color="#d9822b", label="acerto necessário p/ empatar")
axs[0].set_ylim(45, 58); axs[0].legend(); axs[0].set_title("Acerto de direção × acerto necessário (2024)")
axs[1].bar(comp.index, comp["previsto > custo: total R$"].astype(float), color="#3a9a5b")
axs[1].axhline(0, color="gray"); axs[1].set_title("Lucro total em 2024, regra previsto > custo (R$)")
plt.tight_layout(); plt.savefig("passo07_comparacao_candles.png", dpi=120); plt.show()

## Conclusões do Passo 7

*(preenchidas após rodar)*